# 90 — Executive Dashboard Narrator

## Goal
Read **BI metrics** (revenue, churn, pipeline, CSAT) and
**generate a weekly business narrative** — the kind of summary
a CEO reads on Monday morning.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Tool calling** | LLM decides which metrics to fetch |
| **Metric interpretation** | Week-over-week trends, RAG status |
| **Narrative generation** | Business-friendly language from raw numbers |
| **Multi-tool orchestration** | Agent calls several data tools |

## Stack
- `langchain-ollama` — ChatOllama with tool calling
- `langchain-core` — message types and tool schema
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Simulated BI Data

We simulate a data warehouse with weekly KPI snapshots.

In [ ]:
# Simulated BI Data Warehouse
BI_DATA = {
    "revenue": {
        "current_week": 2_450_000, "prior_week": 2_310_000,
        "mtd": 9_200_000, "target_mtd": 10_000_000,
        "top_product": "Enterprise Suite", "top_product_revenue": 980_000,
        "currency": "USD"
    },
    "pipeline": {
        "total_value": 18_500_000, "deals_count": 47,
        "avg_deal_size": 393_617,
        "stage_breakdown": {
            "discovery": {"count": 15, "value": 4_200_000},
            "proposal": {"count": 12, "value": 5_800_000},
            "negotiation": {"count": 8, "value": 4_100_000},
            "closing": {"count": 12, "value": 4_400_000}
        },
        "at_risk_deals": 3, "at_risk_value": 1_200_000
    },
    "customer_metrics": {
        "total_customers": 1_245,
        "new_this_week": 8, "churned_this_week": 3,
        "churn_rate_monthly": 1.8,
        "nps_score": 62, "nps_prior_month": 58,
        "csat_score": 4.3, "csat_max": 5.0,
        "support_tickets_open": 34, "avg_resolution_hours": 18
    },
    "operations": {
        "headcount": 312, "open_positions": 14,
        "avg_time_to_hire_days": 42,
        "server_uptime_pct": 99.94,
        "incidents_this_week": 2,
        "incident_detail": [
            {"id": "INC-445", "severity": "P2", "desc": "API latency spike (Tue 2pm, 45min)"},
            {"id": "INC-446", "severity": "P3", "desc": "Email delivery delay (Wed 9am, 2hr)"}
        ]
    },
    "finance": {
        "burn_rate_monthly": 3_100_000,
        "runway_months": 14.5,
        "gross_margin_pct": 72.3,
        "arr": 28_400_000, "arr_growth_yoy_pct": 34
    }
}

print(f"BI data loaded: {list(BI_DATA.keys())}")

## Step 2 — Define Metric Lookup Tools

In [ ]:
@tool
def get_revenue_metrics() -> str:
    """Get current revenue metrics including weekly, MTD, and top product."""
    return json.dumps(BI_DATA["revenue"], indent=2)

@tool
def get_pipeline_metrics() -> str:
    """Get sales pipeline data: total value, stages, at-risk deals."""
    return json.dumps(BI_DATA["pipeline"], indent=2)

@tool
def get_customer_metrics() -> str:
    """Get customer health metrics: churn, NPS, CSAT, support volume."""
    return json.dumps(BI_DATA["customer_metrics"], indent=2)

@tool
def get_operations_metrics() -> str:
    """Get operational metrics: headcount, uptime, incidents."""
    return json.dumps(BI_DATA["operations"], indent=2)

@tool
def get_finance_metrics() -> str:
    """Get financial metrics: burn rate, runway, ARR, margins."""
    return json.dumps(BI_DATA["finance"], indent=2)

tools = [get_revenue_metrics, get_pipeline_metrics,
         get_customer_metrics, get_operations_metrics, get_finance_metrics]

llm_with_tools = llm.bind_tools(tools)

print(f"Defined {len(tools)} metric tools")
print("Tools bound to LLM")

## Step 3 — Tool-Calling Agent Loop

The LLM reads a prompt, decides which tools to call, receives the
data, and generates a narrative. We'll implement a simple
call → execute → respond loop.

In [ ]:
# Build the tool lookup
tool_map = {t.name: t for t in tools}

def run_dashboard_agent(prompt: str) -> str:
    """Run the tool-calling loop and return the final narrative."""
    messages = [
        SystemMessage(content="""You are an executive dashboard narrator.
Your job: fetch KPI data using the available tools, then write a
polished weekly business narrative (like a Monday morning CEO briefing).

Guidelines:
- Call ALL available metric tools to get comprehensive data
- Use clear section headers: Revenue, Pipeline, Customers, Operations, Finance
- Include week-over-week changes and % where available
- Highlight wins, concerns, and action items
- Keep language executive-friendly (no jargon)
- End with 3 key takeaways

Do NOT wrap your response in thinking tags."""),
        HumanMessage(content=prompt)
    ]
    
    # Step 1: LLM decides which tools to call
    print("📊 Agent deciding which metrics to fetch...")
    response = llm_with_tools.invoke(messages)
    messages.append(response)
    
    # Step 2: Execute tool calls
    if response.tool_calls:
        print(f"   Tools requested: {[tc['name'] for tc in response.tool_calls]}")
        for tc in response.tool_calls:
            tool_name = tc["name"]
            print(f"   ⚙️ Calling {tool_name}...")
            result = tool_map[tool_name].invoke(tc["args"])
            messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        
        # Step 3: LLM generates narrative from tool results
        print("   ✍️ Generating narrative...")
        final = llm_with_tools.invoke(messages)
        text = final.content
    else:
        # No tool calls — LLM responded directly
        text = response.content
    
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

print("Dashboard agent loop defined")

In [ ]:
# Run the weekly narrative
narrative = run_dashboard_agent(
    "Generate the weekly business narrative for the week ending November 10, 2024. "
    "Fetch all available metrics and produce a comprehensive CEO briefing."
)

print("\n" + "=" * 60)
print("WEEKLY BUSINESS NARRATIVE")
print("=" * 60)
print(narrative)

## Step 4 — Focused Follow-Up Queries

The same agent can answer focused questions using the metric tools.

In [ ]:
# Follow-up: pipeline risk analysis
followup = run_dashboard_agent(
    "Give me a focused briefing on pipeline risk. "
    "How many deals are at risk, what's the value exposure, "
    "and what should we do about it?"
)

print("\n" + "=" * 60)
print("PIPELINE RISK BRIEFING")
print("=" * 60)
print(followup)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Tool definition** | `@tool` decorator with docstrings |
| **Tool binding** | `llm.bind_tools(tools)` |
| **Tool execution** | Check `response.tool_calls`, execute, return `ToolMessage` |
| **Agent loop** | Prompt → tool calls → execute → final response |
| **Narrative gen** | LLM interprets raw metrics as business language |

## How Tool Calling Works

```
User: "Generate weekly briefing"
      ↓
LLM: "I need revenue, pipeline, customer, ops, finance data"
      ↓ (tool_calls)
Agent: execute all 5 tools, collect results
      ↓ (ToolMessages)
LLM: Generates polished narrative from raw data
      ↓
User: Reads CEO briefing
```

## Extending This Project
- Connect to real BI tools (Metabase, Looker, PowerBI API)
- Add historical trend comparison (this week vs same week last year)
- Email delivery via SendGrid/SES
- Slack integration for #exec-briefing channel
- Add charts using matplotlib before narrative generation